# Ngrams

N-grams, according to Eder, are inferior to most-frequent word (MWF) analysis of a text, but can be effective with noisy data. The focus of Eder on English means that one other advantage, the efficacy in more inflected languages, shouldn't be overlooked.

First, to retrieve the data. We don't need the pre-tokenized data, since it's not to be parsed.

In [3]:
import pandas as pd

base_data = pd.read_csv("caesar_text_perseus.csv", sep="|")

print(base_data.info())
print("-" * 40)
print(base_data.iloc[1,])

<class 'pandas.DataFrame'>
RangeIndex: 2151 entries, 0 to 2150
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  2151 non-null   int64
 1   work        2151 non-null   str  
 2   book        2151 non-null   int64
 3   text        2151 non-null   str  
 4   src         2151 non-null   str  
 5   chapter     2151 non-null   int64
dtypes: int64(3), str(3)
memory usage: 101.0 KB
None
----------------------------------------
Unnamed: 0                                                    1
work                                                     gallic
book                                                          1
text          Apud Helvetios longe nobilissimus fuit et diti...
src            urn:cts:latinLit:phi0448.phi001.perseus-lat2:1.2
chapter                                                       1
Name: 1, dtype: object


I need to make sure there aren't subtly different characters due to encoding differences.

In [4]:
def unique_ch(input_str):
    collector = []
    for char in input_str:
        if char in collector:
            continue
        collector.append(char)
    return collector

all_text = " ".join(base_data["text"].to_list())
print(sorted(unique_ch(all_text)))

[' ', '!', '"', "'", '(', ')', ',', '-', '.', '/', '0', '2', '3', '5', '6', '7', '8', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'X', 'Z', '[', '\\', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '·', '̅', 'Ι', '—', '‘', '’', '“', '”', '†']


Nothing too special to see there.

Now we need to create a function that allows us to divide up the text in two ways.

One is to do by citation, the other is to further subdivide into equal groups.

In [5]:
import re

# Remove non-alphabetic characters
def clean(input_str):
    val = re.sub(pattern="[^a-zA-Z ]", repl="", string=input_str).casefold()
    return re.sub(" +", " ", val)

print(clean(base_data["text"][0]))

gallia est omnis divisa in partes tres quarum unam incolunt belgae aliam aquitani tertiam qui ipsorum lingua celtae nostra galli appellantur hi omnes lingua institutis legibus inter se differunt gallos ab aquitanis garumna flumen a belgis matrona et sequana dividit horum omnium fortissimi sunt belgae propterea quod a cultu atque humanitate provinciae longissime absunt minimeque ad eos mercatores saepe commeant atque ea quae ad effeminandos animos pertinent important proximique sunt germanis qui trans rhenum incolunt quibuscum continenter bellum gerunt qua de causa helvetii quoque reliquos gallos virtute praecedunt quod fere cotidianis proeliis cum germanis contendunt cum aut suis finibus eos prohibent aut ipsi in eorum finibus bellum gerunt eorum una pars quam gallos obtinere dictum est initium capit a flumine rhodano continetur garumna flumine oceano finibus belgarum attingit etiam ab sequanis et helvetiis flumen rhenum vergit ad septentriones belgae ab extremis galliae finibus oriunt

In [6]:
# get ngrams, keeping word boundaries as a character
def ngram(text, n):
        return [text[i:(i + n)] for i in range(len(text)) if (i + n - 1) < len(text)]

print(ngram("Gallia est omnis divisa in partes tres", 2))
print(ngram("Gallia est omnis divisa in partes tres", 3))

['Ga', 'al', 'll', 'li', 'ia', 'a ', ' e', 'es', 'st', 't ', ' o', 'om', 'mn', 'ni', 'is', 's ', ' d', 'di', 'iv', 'vi', 'is', 'sa', 'a ', ' i', 'in', 'n ', ' p', 'pa', 'ar', 'rt', 'te', 'es', 's ', ' t', 'tr', 're', 'es']
['Gal', 'all', 'lli', 'lia', 'ia ', 'a e', ' es', 'est', 'st ', 't o', ' om', 'omn', 'mni', 'nis', 'is ', 's d', ' di', 'div', 'ivi', 'vis', 'isa', 'sa ', 'a i', ' in', 'in ', 'n p', ' pa', 'par', 'art', 'rte', 'tes', 'es ', 's t', ' tr', 'tre', 'res']


In [7]:
# Utility for next cell

def group_str(text: str, size: int):
    tokenized = text.split(" ")
    n = len(tokenized) // size
    assert n > 0, "size variable is larger than number of words in the provided text segment"

    indices = list(range(0, size * (n + 1), size))
    if len(tokenized) % size != 0:
        indices.append(len(tokenized))

    return [" ".join(tokenized[indices[i]:indices[i + 1]]) for i in range(0, len(indices) - 1)]

test = clean(base_data['text'][0])
grouped_test = group_str(text = test, size = 20)
print(grouped_test)

print(group_str(text = test, size = 178)) # Still works!

# This should throw an error
print(group_str(text = test, size = 179))

['gallia est omnis divisa in partes tres quarum unam incolunt belgae aliam aquitani tertiam qui ipsorum lingua celtae nostra galli', 'appellantur hi omnes lingua institutis legibus inter se differunt gallos ab aquitanis garumna flumen a belgis matrona et sequana dividit', 'horum omnium fortissimi sunt belgae propterea quod a cultu atque humanitate provinciae longissime absunt minimeque ad eos mercatores saepe commeant', 'atque ea quae ad effeminandos animos pertinent important proximique sunt germanis qui trans rhenum incolunt quibuscum continenter bellum gerunt qua', 'de causa helvetii quoque reliquos gallos virtute praecedunt quod fere cotidianis proeliis cum germanis contendunt cum aut suis finibus eos', 'prohibent aut ipsi in eorum finibus bellum gerunt eorum una pars quam gallos obtinere dictum est initium capit a flumine', 'rhodano continetur garumna flumine oceano finibus belgarum attingit etiam ab sequanis et helvetiis flumen rhenum vergit ad septentriones belgae ab', 'extremis

AssertionError: size variable is larger than number of words in the provided text segment

In [8]:
# Function that returns a list of data structures of two components: a string, and a citation.

class Segment:
    def __init__(self, cite, string):
        self.cite = cite
        self.string = string

def get_groups(df, cite: str="book", size: int=0) -> Segment:
    """
    Return groups of cleaned text by book or section, which is then further divided into texts of equal sizes.

    @param df: pandas DataFrame csv
    @param cite: 'book' or 'work'
    @param size: Default of 0 keeps each book/work intact. An integer gives the maximum size each subsection should be, if dividing the text up is desired"""
    
    # Concatenate rows together by grouping (whether book or work)
    # During this loop, add to a Segment with the citation in a list
    grouped = df.groupby(['work', cite])

    citation_segments = [Segment(name, clean(" ".join(group['text'].to_list()))) for name, group in grouped]

    if size > 0:
        final_segments = []
        for s in citation_segments:
            final_segments += [Segment(s.cite, x) for x in group_str(s.string, size)]
        return final_segments
    else:
        return citation_segments


segments = get_groups(base_data, "book")
print(segments[0].cite, "\n", "-" * 100, '\n', segments[0].string[0:400])
print("-" * 100)
print(get_groups(base_data, "book", 10)[0].string)
       
    

('african', 1) 
 ---------------------------------------------------------------------------------------------------- 
 caesar itineribus iustis confectis nullo die intermisso a d xiiii kal ian lilybaeum pervenit statimque ostendit sese navis velle conscendere cum non amplius legionem tironum haberet unam equites vix dc tabernaculum secundum litus ipsum constituit ut prope fluctus verberaret hoc eo consilio fecit ne quis sibi morae quicquam fore speraret et ut omnes in dies horasque parati essent incidit per id tem
----------------------------------------------------------------------------------------------------
caesar itineribus iustis confectis nullo die intermisso a d xiiii
